In [62]:
!pip install chromadb

In [63]:
!pip install pypdf

In [64]:
# IMPORT LIBRARIES
import chromadb

from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from google.colab import files, drive

# UPLOAD PDF
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

# EXTRACT TEXT
reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:

    page_text = page.extract_text()

    if page_text:
        text += page_text + "\n"

print("Total Characters:", len(text))

# TEXT CHUNKING
chunk_size = 500
overlap = 50

chunks = []

start = 0

while start < len(text):

    end = start + chunk_size

    chunks.append(text[start:end])

    start += chunk_size - overlap

print("Total Chunks:", len(chunks))

# GENERATE EMBEDDINGS
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print("Embedding Shape:", embeddings.shape)


Saving Student_Handbook_Semantic_Search_Assignment.pdf to Student_Handbook_Semantic_Search_Assignment (1).pdf
Total Characters: 17441
Total Chunks: 39


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Shape: (39, 384)


In [65]:
# drive mount
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [66]:
path = "/content/drive/MyDrive/RAG and VectorDB - St. John/DAY 3/Database"

In [67]:
client = chromadb.PersistentClient(path=path)

In [68]:
collection = client.get_or_create_collection(
    name="student_admission_handbook",
    metadata={"hnsw:space": "cosine"}
)

In [69]:
collection.upsert(
    ids = [f"chunk_{i}" for i in range(len(chunks))],
    documents = chunks,
    embeddings = embeddings.tolist()
)

In [70]:
print(collection.count())

39


In [73]:
query = input("Enter your query :")


Enter your query :what is the eligibility criteria ?


In [74]:
query_embedding = model.encode([query])

In [75]:
results = collection.query(
    query_embeddings = query_embedding.tolist(),
    n_results = 5,
    include=["documents", "distances"]
)

In [76]:
for i in range(len(results["documents"][0])):
  documents = (results["documents"][0][i])
  distance = (results["distances"][0][i])

  similarity = 1 - distance

  print("Document no :", i+1)
  print ("Content :",documents)
  print ("Distance :", distance)
  print ("Similarity :", similarity)
  print ("-"*50)

Document no : 1
Content :  Meeting minimum eligibility does not guarantee
admission.
Application process
Applicants complete the online form, select a programme, provide accurate details, upload requested documents and pay the
application fee through an approved channel. A submission acknowledgement is generated after the form is received.
Applicants should retain the acknowledgement number.
Documents and selection
Commonly requested documents include a photograph, proof of identity, secondary and higher-secondary mark s
Distance : 0.4102151393890381
Similarity : 0.5897848606109619
--------------------------------------------------
Document no : 2
Content : ission notice
before applying. The following general process is fictional and is included only for educational practice.
Undergraduate eligibility
Applicants to engineering programmes are expected to have completed senior secondary education with Mathematics and
Physics, together with an approved third subject. Applicants to busines

In [77]:
similarity_threshold = 0.50

In [78]:
filtered_results = []

In [79]:
for i in range (len(results["documents"][0])):
  documents = results["documents"][0][i]
  distances = results["distances"][0][i]

  similarity = 1 - distances
  if similarity >= similarity_threshold:
    filtered_results.append(
         documents
    )


In [80]:
print("Relevant Results")
print("RELEVANT DOCUMENTS\n")

for i, document in enumerate(filtered_results):

    print("Document:", i + 1)
    print("Content:", document)
    print("-" * 60)


print("Total Relevant Documents:", len(filtered_results))

Relevant Results
RELEVANT DOCUMENTS

Document: 1
Content:  Meeting minimum eligibility does not guarantee
admission.
Application process
Applicants complete the online form, select a programme, provide accurate details, upload requested documents and pay the
application fee through an approved channel. A submission acknowledgement is generated after the form is received.
Applicants should retain the acknowledgement number.
Documents and selection
Commonly requested documents include a photograph, proof of identity, secondary and higher-secondary mark s
------------------------------------------------------------
Document: 2
Content: ission notice
before applying. The following general process is fictional and is included only for educational practice.
Undergraduate eligibility
Applicants to engineering programmes are expected to have completed senior secondary education with Mathematics and
Physics, together with an approved third subject. Applicants to business programmes may apply fr

In [81]:
context = ""

for i, document in enumerate(filtered_results):

    context +=f"""
    Document {i+1}
    {document}

    ---------------------

    """

print("Final Context")
if context.strip():
      print(context)
else:
     print("No relevant documents found.")


Final Context

    Document 1
     Meeting minimum eligibility does not guarantee
admission.
Application process
Applicants complete the online form, select a programme, provide accurate details, upload requested documents and pay the
application fee through an approved channel. A submission acknowledgement is generated after the form is received.
Applicants should retain the acknowledgement number.
Documents and selection
Commonly requested documents include a photograph, proof of identity, secondary and higher-secondary mark s

    ---------------------

    
    Document 2
    ission notice
before applying. The following general process is fictional and is included only for educational practice.
Undergraduate eligibility
Applicants to engineering programmes are expected to have completed senior secondary education with Mathematics and
Physics, together with an approved third subject. Applicants to business programmes may apply from any recognized stream.
Some applied-science program

In [82]:
system_instruction = """
    You are a helpful college infomation assistant.

    Follow these rules :
    1. Answer only using provided context.
    2. Do not use external knowledge.
    3. DO NOT make up information.
    4. If the answer iis not available in context
    say " Sorry, i dont have enough information.PLease contact Authoritize."


    Give real and simple answer.

"""

In [83]:

if not context.strip():
  print("No relevant context available")

else:
  prompt = f"""
  CONTEXT : {context},
  QUESTION : {query}

  """

print("SYTEM INSTRUCTIONS -")
print(system_instruction)
print("-"*50)
print("PROMPT -")
print(prompt)

SYTEM INSTRUCTIONS -

    You are a helpful college infomation assistant.

    Follow these rules :
    1. Answer only using provided context.
    2. Do not use external knowledge.
    3. DO NOT make up information.
    4. If the answer iis not available in context 
    say " Sorry, i dont have enough information.PLease contact Authoritize."
    

    Give real and simple answer. 


--------------------------------------------------
PROMPT -

  CONTEXT : 
    Document 1
     Meeting minimum eligibility does not guarantee
admission.
Application process
Applicants complete the online form, select a programme, provide accurate details, upload requested documents and pay the
application fee through an approved channel. A submission acknowledgement is generated after the form is received.
Applicants should retain the acknowledgement number.
Documents and selection
Commonly requested documents include a photograph, proof of identity, secondary and higher-secondary mark s

    ---------------

In [84]:
from google.colab import userdata
from google import genai

In [85]:
key = userdata.get("GEMINI_API_KEY")

In [87]:
gemini_client = genai.Client(api_key=key)
print("Gemini connected successfuly")

Gemini connected successfuly


In [88]:
response =gemini_client.models.generate_content(
    model = "gemini-3.8-flash",
    contents = prompt,
    config = {
        "system_instruction" : system_instruction,
        "max_output_tokens" : 1000,
        "temperature" : 0.3
    }
)

In [89]:
print("QUERY :")
print(query)
print("-----")
print("RESPONSE :")
print(response.text)


QUERY :
what is the eligibility criteria ?
-----
RESPONSE :
Based on the provided context, the undergraduate eligibility criteria are:

* **Engineering programmes:** Applicants must have completed senior secondary education with Mathematics and Physics, together with an approved third subject.
* **Business programmes:** Applicants may apply from any recognized stream.
* **Applied-science programmes:** Some programmes may require Biology or Chemistry.

*(Note: Meeting the minimum eligibility does not guarantee admission.)*
